In [11]:

# Merge the output together
import pandas as pd
import os

# Create a new text file
with open("merged_output.txt", "w") as merged_file:

  # Iterate over the list of text files
  for i in range(1, 22):
    print(i)
    # Read the contents of the text file
    with open("./outputs/output_"+str(i)+"/output/sample_disPred.txt", "r") as file:
      contents = file.read()

    # Write the contents of the text file to the merged text file
    merged_file.write(contents)
    
merged_file.close()
 


1
2
3
4
5
6
7
8
9
10
11
12
13
14
15
16
17
18
19
20
21


In [12]:
mapdf=pd.read_csv("./inputs/mapfile.txt",header=None,index_col=1)
mapdf
# mapid= mapdf.loc[" "+"Col4a1PA"][0]
# mapid

,0
1,
Col4a1PA,Col4a1-PA
Col4a1PB,Col4a1-PB
Col4a1PC,Col4a1-PC
MpPB,Mp-PB
MpPE,Mp-PE
...,...
Cp7FcPA,Cp7Fc-PA
PxdPA,Pxd-PA
PxdPB,Pxd-PB


In [13]:
maplengthdf=pd.read_csv("./inputs/proteinLengths.txt",header=None,index_col=0)
maplengthdf
# protein_idM="MpPB"
# print(protein_idM)
# maplengthdf= int(maplengthdf.loc[protein_idM])
# maplengthdf


,1
0,
Col4a1PA,1779
Col4a1PB,1779
Col4a1PC,1779
MpPB,1008
MpPE,778
...,...
Cp7FcPA,321
PxdPA,690
PxdPB,690


In [14]:
maplengthdf=pd.read_csv("./inputs/proteinLengths.txt",header=None,index_col=0)
mapdf=pd.read_csv("./inputs/mapfile.txt",header=None,index_col=1)

df=pd.DataFrame(columns=['Protein_ID', "Number of Amino Acids",'Order/Disordered','Disordered Regions (%)'])
df2=pd.DataFrame(columns=['Protein_ID', "Total Number of Amino Acids",'Disordered Regions',"Number of Disordered Residues","Confidence Score"])
count =0
order_disorder=0
order_disordercount=0
totalnumberofproteins=0
totalnumberofdisorderproteins=0

flag =0
sf=0
startIndex=0
endIndex=0
confidenceScore=0.0

outputfile=open('output.txt','w')
for line in open('merged_output.txt','r'):
    if line.startswith('>'):
        if sf==1:
            sf=0
            endIndex=startIndex
            
            print(startIndex,endIndex,endIndex-startIndex+1 ,round(confidenceScore/(endIndex-startIndex+1), 2),file=outputfile)       
            df2=df2.append({'Protein_ID':protein_id,"Total Number of Amino Acids": int(maplengthdf.loc[protein_idM] ), 'Disordered Regions':str(startIndex)+" - "+str(endIndex),"Number of Disordered Residues":endIndex-startIndex+1,'Confidence Score':round(confidenceScore/(endIndex-startIndex+1), 3)},ignore_index=True)       
     
            confidenceScore=0
        
        print(line,file=outputfile) 
       
        if flag == 0:
            flag =1
        else:
            # print(protein_id, count ,order_disorder, order_disordercount)  
            if  order_disordercount == 0:
                order_disorderpercentage=0
            else: 
                order_disorderpercentage=order_disordercount/count*100   
                
            df=df.append({'Protein_ID':protein_id,"Number of Amino Acids":count , 'Order/Disordered':order_disorder,'Disordered Regions (%)':order_disorderpercentage},ignore_index=True)
            
            totalnumberofproteins+=1
            if(order_disorder==1):
                totalnumberofdisorderproteins+=1
            count =0
            order_disorder=0
            order_disordercount=0
        
        protein_idM=line[1:-1] 
        
        protein_id= mapdf.loc[" "+protein_idM][0]

        
    else:
        count+=1
        line=line.split()
        # print(line)

        if line[-1]=='1':
            if sf==0:
                sf=1
                startIndex=int(line[0])
            confidenceScore=confidenceScore+float(line[2])
            order_disorder=1
            order_disordercount+=1
        else:
            if sf==1:
                sf=0
                endIndex=int(line[0])-1   
                
                print(startIndex,endIndex,endIndex-startIndex+1 ,round(confidenceScore/(endIndex-startIndex+1), 2),file=outputfile)  
                df2=df2.append({'Protein_ID':protein_id,"Total Number of Amino Acids": int(maplengthdf.loc[protein_idM] ), 'Disordered Regions':str(startIndex)+" - "+str(endIndex),"Number of Disordered Residues":endIndex-startIndex+1,'Confidence Score':round(confidenceScore/(endIndex-startIndex+1), 3)},ignore_index=True)       
                confidenceScore=0
                
            
# print(protein_id, count ,order_disorder, order_disordercount)  
if  order_disordercount == 0:
    order_disorderpercentage=0
else: 
    order_disorderpercentage=order_disordercount/count*100   
    
df=df.append({'Protein_ID':protein_id,"Number of Amino Acids":count , 'Order/Disordered':order_disorder,'Disordered Regions (%)':order_disorderpercentage},ignore_index=True)

totalnumberofproteins+=1
if(order_disorder==1):
    totalnumberofdisorderproteins+=1



outputfile.close()

print(totalnumberofproteins,totalnumberofdisorderproteins)
df.sort_values(by=['Disordered Regions (%)'],ascending=False,inplace=True)
df.to_csv('Results_V2.csv',index=False)

1246 1134


In [15]:
df2
df2.to_csv('Results_V3.csv',index=False)